In [ ]:
"""
=============================================================================
 Multi-Stage Attack Dataset Builder  v2
 ----------------------------------------
 PERUBAHAN dari v1:
   1. Sequence building SEBELUM sampling (bukan sesudah)
      → tidak ada lagi chain yang putus karena rows dibuang duluan
   2. Dual perspective: evaluasi sequence dari dst_ip DAN src_ip
      → IP 192.168.1.30 sbg attacker (src) juga terdeteksi 1→2→3
   3. Keep ALL rows dari IP yang punya valid sequence
      → tidak ada event yang hilang dari chain
   4. Naikkan quota normal & single untuk capai 200K+
   5. Top-up sampling setelah sequence preservation

CARA MENJALANKAN:
   1. Ubah DATA_DIR ke path folder berisi 23 CSV
   2. python build_multistage_dataset_v2.py
   3. Output: train_test_network.csv + dataset_summary.txt
=============================================================================
"""

import pandas as pd
import numpy as np
import glob
import os
from collections import defaultdict
from typing import Optional, List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# ====================================================================
# CONFIG
# ====================================================================

DATA_DIR = r"Processed_Network_dataset"   # <- ubah sesuai lokasi CSV kamu
OUTPUT_FILE = "train_test_network.csv"
SUMMARY_FILE = "dataset_summary.txt"

SEED = 42

STAGE_MAP = {
    'normal':     0,
    'scanning':   1,
    'xss':        2,  'ddos': 2, 'dos': 2,
    'injection':  2,  'mitm': 2, 'password': 2,
    'backdoor':   3,
    'ransomware': 3,
}

# ---- Sampling targets ----
# Jumlah MINIMUM rows per attack type yang akan di-keep.
# Jika stok di data asli kurang, ambil semua yang ada.
SAMPLE_PER_TYPE = 30_000
NORMAL_TARGET   = 300_000    # normal traffic quota (besar supaya > 200K total)

# Sequence priority weights (untuk reporting, bukan filtering)
# Di v2 kita keep SEMUA valid sequence, tidak di-cap.
SEQUENCE_PRIORITY = ['1-2-3', '1-2', '1-3', '2-3']

TS_COL = 'ts'


# ====================================================================
# STEP 1: LOAD & MERGE
# ====================================================================

def load_and_merge(data_dir: str) -> pd.DataFrame:
    """Load semua Network_dataset_*.csv dan gabungkan."""
    print("=" * 60)
    print("STEP 1: Loading CSV files")
    print("=" * 60)

    patterns = [
        os.path.join(data_dir, "Network_dataset_*.csv"),
        os.path.join(data_dir, "Network_dataset*.csv"),
    ]

    all_files = []
    for pat in patterns:
        all_files = sorted(glob.glob(pat))
        if all_files:
            break

    if not all_files:
        raise FileNotFoundError(
            f"Tidak ditemukan CSV di '{data_dir}'. Pastikan DATA_DIR benar."
        )

    frames = []
    for i, f in enumerate(all_files, 1):
        try:
            chunk = pd.read_csv(f, low_memory=False)
            frames.append(chunk)
            print(f"  [{i:2d}/{len(all_files)}] {os.path.basename(f)}: "
                  f"{len(chunk):>10,} rows")
        except Exception as e:
            print(f"  [{i:2d}/{len(all_files)}] ERROR {os.path.basename(f)}: {e}")

    df = pd.concat(frames, ignore_index=True)
    print(f"\n  Total merged: {len(df):,} rows, {df.shape[1]} cols")
    return df


# ====================================================================
# STEP 2: CLEAN & STAGE MAP
# ====================================================================

def clean_and_map_stages(df: pd.DataFrame) -> pd.DataFrame:
    """Bersihkan data dan tambah kolom 'stage'."""
    print("\n" + "=" * 60)
    print("STEP 2: Cleaning & Stage Mapping")
    print("=" * 60)

    initial = len(df)

    # Normalisasi type
    df['type'] = df['type'].astype(str).str.strip().str.lower()

    # Cari kolom timestamp
    if TS_COL not in df.columns:
        for alt in ['timestamp', 'waktu', 'time']:
            if alt in df.columns:
                df = df.rename(columns={alt: TS_COL})
                break

    # Drop NaN pada kolom kritis
    critical = [c for c in [TS_COL, 'src_ip', 'dst_ip', 'type'] if c in df.columns]
    df = df.dropna(subset=critical)

    # Timestamp ke numeric
    df[TS_COL] = pd.to_numeric(df[TS_COL], errors='coerce')
    df = df.dropna(subset=[TS_COL])

    # Drop exact duplicates
    df = df.drop_duplicates()

    # Stage mapping
    df['stage'] = df['type'].map(STAGE_MAP)
    unmapped = df['stage'].isna().sum()
    if unmapped > 0:
        bad_types = df.loc[df['stage'].isna(), 'type'].unique()
        print(f"  WARNING: {unmapped:,} rows type tidak dikenali: {bad_types}")
        df = df.dropna(subset=['stage'])
    df['stage'] = df['stage'].astype(int)

    print(f"  {initial:,} -> {len(df):,} rows (dropped {initial - len(df):,})")
    print(f"\n  Type distribution:")
    for t, c in df['type'].value_counts().items():
        print(f"    {t:15s}: {c:>10,}  (stage {STAGE_MAP.get(t, '?')})")

    return df


# ====================================================================
# STEP 3: BUILD SEQUENCES (sebelum sampling!)
# ====================================================================

def build_sequences(df: pd.DataFrame) -> pd.DataFrame:
    """
    Evaluasi sequence multi-stage dari DUA perspektif:
      - dst_ip: IP target menerima scanning → exploit → post-exploit
      - src_ip: IP attacker melakukan scanning → exploit → post-exploit

    Setiap row ditandai dengan sequence_type terbaik yang ditemukan.
    """
    print("\n" + "=" * 60)
    print("STEP 3: Building Multi-Stage Sequences (dual perspective)")
    print("=" * 60)

    df = df.sort_values(TS_COL).reset_index(drop=True)

    # Inisialisasi kolom
    df['seq_dst'] = 'none'     # sequence dari perspektif dst_ip
    df['seq_src'] = 'none'     # sequence dari perspektif src_ip

    attack = df[df['stage'] > 0]

    # --- Perspektif 1: dst_ip (target) ---
    print("\n  [dst_ip perspective] Analyzing sequences...")
    dst_results = _analyze_sequences_by_ip(attack, ip_col='dst_ip')
    for seq_type, ip_set in dst_results.items():
        for ip in ip_set:
            mask = (df['dst_ip'] == ip) & (df['stage'] > 0)
            df.loc[mask, 'seq_dst'] = seq_type

    # --- Perspektif 2: src_ip (attacker) ---
    print("  [src_ip perspective] Analyzing sequences...")
    src_results = _analyze_sequences_by_ip(attack, ip_col='src_ip')
    for seq_type, ip_set in src_results.items():
        for ip in ip_set:
            mask = (df['src_ip'] == ip) & (df['stage'] > 0)
            df.loc[mask, 'seq_src'] = seq_type

    # --- Gabungkan: ambil sequence terbaik dari kedua perspektif ---
    priority = {'1-2-3': 4, '1-2': 3, '1-3': 2, '2-3': 1, 'single': 0, 'none': -1}

    def best_seq(row):
        d = row['seq_dst']
        s = row['seq_src']
        if priority.get(d, -1) >= priority.get(s, -1):
            return d
        return s

    df['sequence_type'] = df.apply(best_seq, axis=1)

    # Normal traffic
    df.loc[df['stage'] == 0, 'sequence_type'] = 'normal'

    # Attack tanpa sequence apapun → single
    df.loc[
        (df['stage'] > 0) & (df['sequence_type'].isin(['none'])),
        'sequence_type'
    ] = 'single'

    # Print stats
    print(f"\n  Sequence distribution (combined best):")
    for st, cnt in df['sequence_type'].value_counts().items():
        print(f"    {st:10s}: {cnt:>10,}")

    print(f"\n  dst_ip perspective:")
    for st, cnt in df['seq_dst'].value_counts().items():
        if st != 'none':
            print(f"    {st:10s}: {cnt:>10,}")

    print(f"\n  src_ip perspective:")
    for st, cnt in df['seq_src'].value_counts().items():
        if st != 'none':
            print(f"    {st:10s}: {cnt:>10,}")

    return df


def _analyze_sequences_by_ip(
    attack_df: pd.DataFrame,
    ip_col: str
) -> Dict[str, set]:
    """
    Untuk setiap unique IP di ip_col, cek apakah ada valid multi-stage sequence.
    Return dict: sequence_type -> set of IPs
    """
    results = defaultdict(set)
    grouped = attack_df.groupby(ip_col)
    total = len(grouped)

    for idx, (ip, group) in enumerate(grouped):
        if idx % 10000 == 0 and idx > 0:
            print(f"    {idx:,}/{total:,} IPs processed...")

        group = group.sort_values(TS_COL)
        stages = set(group['stage'].unique())

        s1 = group[group['stage'] == 1]
        s2 = group[group['stage'] == 2]
        s3 = group[group['stage'] == 3]

        assigned = False

        # Prioritas 1: full chain 1→2→3
        if {1, 2, 3} <= stages:
            if _valid_chain(s1, s2, s3):
                results['1-2-3'].add(ip)
                assigned = True

        # Prioritas 2: 1→2
        if not assigned and {1, 2} <= stages:
            if _valid_pair(s1, s2):
                results['1-2'].add(ip)
                assigned = True

        # Prioritas 3: 1→3
        if not assigned and {1, 3} <= stages:
            if _valid_pair(s1, s3):
                results['1-3'].add(ip)
                assigned = True

        # Prioritas 4: 2→3
        if not assigned and {2, 3} <= stages:
            if _valid_pair(s2, s3):
                results['2-3'].add(ip)
                assigned = True

        if not assigned:
            results['single'].add(ip)

    for k in ['1-2-3', '1-2', '1-3', '2-3', 'single']:
        print(f"    {ip_col} {k:10s}: {len(results.get(k, set())):,} IPs")

    return results


def _valid_chain(s1: pd.DataFrame, s2: pd.DataFrame, s3: pd.DataFrame) -> bool:
    """Cek apakah ada event s1 < s2 < s3 secara timestamp."""
    s1_min = s1[TS_COL].min()
    s3_max = s3[TS_COL].max()

    # Ada s2 yang waktunya antara s1 pertama dan s3 terakhir?
    valid_s2 = s2[(s2[TS_COL] > s1_min) & (s2[TS_COL] < s3_max)]
    if len(valid_s2) == 0:
        # Relaxed: minimal s1_min < s2_min
        valid_s2 = s2[s2[TS_COL] > s1_min]

    return len(valid_s2) > 0


def _valid_pair(earlier: pd.DataFrame, later: pd.DataFrame) -> bool:
    """Cek apakah ada event earlier yang ts-nya < salah satu event later."""
    return earlier[TS_COL].min() < later[TS_COL].max()


# ====================================================================
# STEP 4: SMART SAMPLING (setelah sequence, bukan sebelum!)
# ====================================================================

def smart_sampling(df: pd.DataFrame) -> pd.DataFrame:
    """
    Strategi sampling:
      1. KEEP SEMUA rows dari IP yang punya valid sequence (1-2-3, 1-2, 1-3, 2-3)
      2. Top-up dari single-stage attacks sampai target per-type terpenuhi
      3. Top-up normal traffic sampai NORMAL_TARGET
      4. Pastikan IP diversity tetap tinggi
    """
    print("\n" + "=" * 60)
    print("STEP 4: Smart Sampling (sequence-preserving)")
    print("=" * 60)

    rng = np.random.RandomState(SEED)

    # ---- Part A: Keep semua rows dari valid sequences ----
    valid_seqs = ['1-2-3', '1-2', '1-3', '2-3']
    df_sequenced = df[df['sequence_type'].isin(valid_seqs)].copy()
    print(f"\n  A) Sequenced rows (keep all): {len(df_sequenced):,}")

    # Hitung berapa per type sudah terkumpul dari sequenced data
    type_counts = df_sequenced['type'].value_counts().to_dict()
    print(f"     Per type dari sequenced:")
    for t in sorted(STAGE_MAP.keys()):
        if t != 'normal':
            print(f"       {t:15s}: {type_counts.get(t, 0):>8,}")

    # ---- Part B: Top-up single-stage attacks ----
    df_single = df[df['sequence_type'] == 'single'].copy()
    topup_frames = []

    print(f"\n  B) Top-up dari single-stage attacks:")
    for attack_type in sorted(STAGE_MAP.keys()):
        if attack_type == 'normal':
            continue

        already = type_counts.get(attack_type, 0)
        needed = max(0, SAMPLE_PER_TYPE - already)

        available = df_single[df_single['type'] == attack_type]

        if needed > 0 and len(available) > 0:
            take = min(needed, len(available))
            sampled = _ip_diverse_sample(available, take, rng)
            topup_frames.append(sampled)
            print(f"    {attack_type:15s}: sudah {already:>6,} + topup {len(sampled):>6,}"
                  f" = {already + len(sampled):>6,}")
        else:
            print(f"    {attack_type:15s}: sudah {already:>6,}, no topup needed")

    # ---- Part C: Normal traffic ----
    df_normal = df[df['stage'] == 0].copy()
    normal_take = min(NORMAL_TARGET, len(df_normal))
    df_normal_sampled = _ip_diverse_sample(df_normal, normal_take, rng)
    print(f"\n  C) Normal traffic: {len(df_normal_sampled):,} / {len(df_normal):,} available")

    # ---- Combine ----
    parts = [df_sequenced, df_normal_sampled] + topup_frames
    result = pd.concat(parts, ignore_index=True)

    # Deduplicate (in case IP overlap caused double-counting)
    before_dedup = len(result)
    result = result.drop_duplicates()
    if before_dedup > len(result):
        print(f"\n  Dedup: {before_dedup:,} -> {len(result):,}")

    print(f"\n  TOTAL after sampling: {len(result):,}")
    return result


def _ip_diverse_sample(
    df: pd.DataFrame, target: int, rng: np.random.RandomState
) -> pd.DataFrame:
    """Sample rows dengan prioritas IP diversity (round-robin per dst_ip)."""
    if len(df) <= target:
        return df

    unique_ips = df['dst_ip'].unique()
    rng.shuffle(unique_ips)

    per_ip = max(1, target // len(unique_ips))

    parts = []
    used_idx = set()

    for ip in unique_ips:
        ip_data = df[df['dst_ip'] == ip]
        n = min(per_ip, len(ip_data))
        sampled = ip_data.sample(n=n, random_state=SEED)
        parts.append(sampled)
        used_idx.update(sampled.index)

    result = pd.concat(parts, ignore_index=True)

    # Fill remaining
    if len(result) < target:
        remaining = df[~df.index.isin(used_idx)]
        extra = min(target - len(result), len(remaining))
        if extra > 0:
            result = pd.concat([
                result,
                remaining.sample(n=extra, random_state=SEED)
            ], ignore_index=True)

    if len(result) > target:
        result = result.sample(n=target, random_state=SEED)

    return result


# ====================================================================
# STEP 5: FINALIZE
# ====================================================================

def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """Sort by timestamp, validasi, print stats."""
    print("\n" + "=" * 60)
    print("STEP 5: Finalizing")
    print("=" * 60)

    # Drop helper columns
    for col in ['seq_dst', 'seq_src']:
        if col in df.columns:
            df = df.drop(columns=[col])

    # Sort by timestamp (penting untuk time-splitting)
    df = df.sort_values(TS_COL).reset_index(drop=True)
    df['stage'] = df['stage'].astype(int)
    if 'label' in df.columns:
        df['label'] = df['label'].astype(int)

    # ---- Stats ----
    n = len(df)
    print(f"\n  Total rows:    {n:,}")
    print(f"  Unique src_ip: {df['src_ip'].nunique():,}")
    print(f"  Unique dst_ip: {df['dst_ip'].nunique():,}")
    print(f"  All unique IP: {pd.concat([df['src_ip'], df['dst_ip']]).nunique():,}")
    print(f"  Timestamp:     {df[TS_COL].min():.0f} - {df[TS_COL].max():.0f}")
    print(f"  Sorted by ts:  {df[TS_COL].is_monotonic_increasing}")

    print(f"\n  Stage distribution:")
    for s, cnt in df['stage'].value_counts().sort_index().items():
        print(f"    Stage {s}: {cnt:>8,} ({cnt/n*100:5.1f}%)")

    print(f"\n  Type distribution:")
    for t, cnt in df['type'].value_counts().items():
        print(f"    {t:15s}: {cnt:>8,} ({cnt/n*100:5.1f}%)")

    print(f"\n  Sequence distribution:")
    for st, cnt in df['sequence_type'].value_counts().items():
        print(f"    {st:10s}: {cnt:>8,} ({cnt/n*100:5.1f}%)")

    # ---- Validasi: cek contoh multi-stage IP ----
    print(f"\n  Contoh IP dengan sequence 1-2-3 (dst_ip):")
    seq123 = df[df['sequence_type'] == '1-2-3']
    for ip in seq123['dst_ip'].unique()[:5]:
        ip_data = seq123[seq123['dst_ip'] == ip]
        stages = sorted(ip_data['stage'].unique())
        print(f"    {ip}: {len(ip_data)} rows, stages={stages}")

    return df


def write_summary(df: pd.DataFrame, path: str):
    """Tulis ringkasan statistik."""
    n = len(df)
    with open(path, 'w') as f:
        f.write("=" * 60 + "\n")
        f.write("DATASET SUMMARY — Multi-Stage Attack Detection v2\n")
        f.write("=" * 60 + "\n\n")

        f.write(f"Total rows:    {n:,}\n")
        f.write(f"Columns:       {df.shape[1]}\n")
        f.write(f"Unique src_ip: {df['src_ip'].nunique():,}\n")
        f.write(f"Unique dst_ip: {df['dst_ip'].nunique():,}\n")
        f.write(f"Timestamp:     {df[TS_COL].min():.0f} - {df[TS_COL].max():.0f}\n\n")

        f.write("Stage Distribution:\n")
        for s, cnt in df['stage'].value_counts().sort_index().items():
            f.write(f"  Stage {s}: {cnt:>8,} ({cnt/n*100:5.1f}%)\n")

        f.write("\nType Distribution:\n")
        for t, cnt in df['type'].value_counts().items():
            f.write(f"  {t:15s}: {cnt:>8,} ({cnt/n*100:5.1f}%)\n")

        f.write("\nSequence Distribution:\n")
        for st, cnt in df['sequence_type'].value_counts().items():
            f.write(f"  {st:10s}: {cnt:>8,} ({cnt/n*100:5.1f}%)\n")

        # Time splitting suggestion
        f.write("\n\nTime Splitting Suggestion:\n")
        for pct in [0.7, 0.8]:
            ts_split = df[TS_COL].quantile(pct)
            n_train = (df[TS_COL] <= ts_split).sum()
            n_test  = (df[TS_COL] > ts_split).sum()
            f.write(f"  {pct:.0%} split at ts={ts_split:.0f}: "
                    f"train={n_train:,} / test={n_test:,}\n")

        # IP splitting info
        f.write("\nIP Splitting Info:\n")
        all_ips = pd.concat([df['src_ip'], df['dst_ip']]).unique()
        f.write(f"  Total unique IPs: {len(all_ips):,}\n")

        f.write("\nColumns:\n")
        for col in df.columns:
            f.write(f"  - {col} ({df[col].dtype})\n")

    print(f"\n  Summary -> {path}")


# ====================================================================
# MAIN PIPELINE
# ====================================================================

def main():
    print("\n" + "#" * 60)
    print("#  Multi-Stage Attack Dataset Builder v2")
    print("#" * 60)

    df = load_and_merge(DATA_DIR)
    df = clean_and_map_stages(df)
    df = build_sequences(df)          # <-- sequences FIRST
    df = smart_sampling(df)            # <-- sampling AFTER
    df = finalize(df)

    # Save
    print("\n" + "=" * 60)
    print("STEP 6: Saving")
    print("=" * 60)

    df.to_csv(OUTPUT_FILE, index=False)
    size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
    print(f"  {OUTPUT_FILE}: {len(df):,} rows, {size_mb:.1f} MB")

    write_summary(df, SUMMARY_FILE)

    print("\n" + "#" * 60)
    print(f"#  DONE — {len(df):,} rows")
    print("#" * 60 + "\n")

    return df


if __name__ == "__main__":
    main()


############################################################
#  Multi-Stage Attack Dataset Builder v2
############################################################
STEP 1: Loading CSV files
